<a href="https://colab.research.google.com/github/fairycookies/Hackton_BPS/blob/main/visdat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re, csv, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
FOLDER_DATA = Path("/content/drive/MyDrive/dataset")
FOLDER_OUTPUT = Path("/content/output")
FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
if not list(FOLDER_DATA.glob("Penduduk Berumur 15 Tahun*.csv")):
    for z in FOLDER_DATA.glob("*.zip"):
        zipfile.ZipFile(z).extractall("/content/_ekstrak")
    hasil = list(Path("/content/_ekstrak").rglob("Penduduk Berumur 15 Tahun*.csv"))
    if hasil:
        FOLDER_DATA = hasil[0].parent

In [ ]:
daftar_drive = list(FOLDER_DATA.glob("*.csv"))
if len(daftar_drive) < 9:
    FOLDER_DATA = Path("/content")

daftar = sorted(FOLDER_DATA.glob("*.csv"))
print("Folder data:", FOLDER_DATA)
print("Jumlah CSV :", len(daftar))
for f in daftar:
    print("  -", f.name)
assert len(daftar) >= 9, "CSV belum lengkap, upload 9 file CSV BPS dulu"

In [ ]:
DIM_PENDIDIKAN = pd.DataFrame([
    ("P01", "Tidak/belum pernah sekolah", "Tidak sekolah",     1, "K1", "Y1"),
    ("P02", "Tidak/belum tamat SD",       "Tidak tamat SD",    2, "K1", "Y1"),
    ("P03", "SD",                         "SD",                3, "K1", "Y2"),
    ("P04", "SLTP",                       "SMP",               4, "K2", "Y3"),
    ("P05", "SLTA Umum/SMU",              "SMA",               5, "K3", "Y4"),
    ("P06", "SLTA Kejuruan/SMK",          "SMK",               6, "K4", "Y4"),
    ("P07", "Akademi/Diploma",            "Diploma I/II/III",  7, "K5", "Y5"),
    ("P08", "Universitas",                "D4/S1/S2/S3",       8, "K6", "Y5"),
], columns=["kode_pendidikan", "nama_bps", "nama_singkat", "urutan",
            "kode_kelompok", "kode_pemuda"])

In [ ]:
DIM_KELOMPOK = pd.DataFrame([
    ("K1", "SD ke bawah",          1, "Pendidikan Dasar"),
    ("K2", "SMP",                  2, "Pendidikan Dasar"),
    ("K3", "SMA",                  3, "Pendidikan Menengah"),
    ("K4", "SMK",                  4, "Pendidikan Menengah"),
    ("K5", "Diploma I/II/III",     5, "Pendidikan Tinggi"),
    ("K6", "Diploma IV/S1/S2/S3",  6, "Pendidikan Tinggi"),
], columns=["kode_kelompok", "kelompok_pendidikan", "urutan", "jenjang"])

In [ ]:
DIM_PEMUDA = pd.DataFrame([
    ("Y1", "Tidak pernah sekolah/Tidak tamat SD", 1),
    ("Y2", "SD/sederajat",                        2),
    ("Y3", "SMP/sederajat",                       3),
    ("Y4", "SMA/SMK/sederajat",                   4),
    ("Y5", "Pendidikan tinggi",                   5),
], columns=["kode_pemuda", "kelompok_pemuda", "urutan"])

In [ ]:
MAP_NAMA          = dict(zip(DIM_PENDIDIKAN.nama_bps, DIM_PENDIDIKAN.kode_pendidikan))
MAP_KELOMPOK_2025 = dict(zip(DIM_KELOMPOK.kelompok_pendidikan, DIM_KELOMPOK.kode_kelompok))
MAP_PEMUDA        = dict(zip(DIM_PEMUDA.kelompok_pemuda, DIM_PEMUDA.kode_pemuda))
BULAN = {"Februari": 2, "Agustus": 8}

In [ ]:
def ke_angka(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s in ("", "-", "—", "..."):
        return np.nan
    return float(s)

In [ ]:
def id_periode(tahun, periode):
    return int(tahun) * 100 + BULAN[str(periode).strip().capitalize()]

In [ ]:

for nama in ["Angkatan_Kerja", "TPT_Pemuda", "Pelatihan_Bersertifikat"]:
    f = next(FOLDER_DATA.glob(f"*{nama}*.csv"))
    df = pd.read_csv(f)
    df["periode_key"] = [id_periode(t, p) for t, p in zip(df.Tahun, df.Periode)]
    print(f.name, "->", df.periode_key.unique())

tahun, periode = "2008 Februari".split()
print(id_periode(tahun, periode))   # 200802

for f in sorted(FOLDER_DATA.glob("Pengangguran Terbuka*.csv")):
    tahun = re.search(r"20\d\d", f.name).group()
    print(f.name, "->", [id_periode(tahun, b) for b in ["Februari", "Agustus"]])

In [ ]:
# Membaca CSV 2025 setelah pustaka diimpor
ak25 = pd.read_csv(next(FOLDER_DATA.glob("*Angkatan_Kerja*.csv")))
tpt_pmd = pd.read_csv(next(FOLDER_DATA.glob("*TPT_Pemuda*.csv")))
pel_pmd = pd.read_csv(next(FOLDER_DATA.glob("*Pelatihan_Bersertifikat*.csv")))

UNPIVOT

In [ ]:
def unpivot_pengangguran(f):
    df = pd.read_csv(f, skiprows=3).drop(columns="Tahunan")
    df = df.rename(columns={df.columns[0]: "nama_bps"})
    df = df.melt(id_vars="nama_bps", var_name="periode", value_name="pengangguran")
    tahun = re.search(r"20\d\d", f.name).group()
    df["periode_key"] = [id_periode(tahun, p) for p in df.periode]
    df["pengangguran"] = df.pengangguran.map(ke_angka)
    return df.drop(columns="periode")

peng = pd.concat([unpivot_pengangguran(f) for f in sorted(FOLDER_DATA.glob("Pengangguran Terbuka*.csv"))],
                 ignore_index=True)
peng.head()

In [ ]:
KOLOM = ["bekerja", "pengangguran", "angkatan_kerja", "pct_bekerja_bps", "bak_sekolah",
         "bak_mengurus_rt", "bak_lainnya", "bukan_angkatan_kerja", "penduduk_15plus", "tpak_bps"]

baris = list(csv.reader(open(next(FOLDER_DATA.glob("Penduduk Berumur 15*.csv")), encoding="utf-8-sig")))
data = []
for i, r in enumerate(baris):
    if r and r[0].startswith("Pendidikan Tertinggi"):
        for awal in (1, 11):
            tahun, periode = r[awal].split()
            for d in baris[i + 3:]:
                if not d or not d[0].strip():
                    break
                data.append([id_periode(tahun, periode), d[0].strip()] +
                            [ke_angka(v) for v in d[awal:awal + 10]])

lebar = pd.DataFrame(data, columns=["periode_key", "nama_bps"] + KOLOM)
peng["nama_bps"] = peng.nama_bps.str.strip()
print(len(lebar), len(peng))   # harus 340 dan 90

In [ ]:
total_lebar = lebar[lebar.nama_bps == "Total"]
total_peng  = peng[peng.nama_bps == "Total"]
lebar = lebar[~lebar.nama_bps.isin(["Total", "Tak Terjawab"])].copy()
peng  = peng[peng.nama_bps != "Total"].copy()

lebar["kode_pendidikan"] = lebar.nama_bps.map(MAP_NAMA)
peng["kode_pendidikan"]  = peng.nama_bps.map(MAP_NAMA)
lebar["bak_sekolah"] = lebar.bak_sekolah.fillna(0)

In [ ]:
tak_terjawab = lebar[lebar.nama_bps == "Tak Terjawab"][KOLOM]
print("Total nilai Tak Terjawab:", tak_terjawab.fillna(0).sum().sum())   # 0.0

In [ ]:
print("Belum terpetakan:", lebar[lebar.kode_pendidikan.isna()].nama_bps.unique(),
      peng[peng.kode_pendidikan.isna()].nama_bps.unique())

In [ ]:
ADA = [k for k in KOLOM if k in lebar.columns]
print("Tipe data:", lebar[ADA].dtypes.unique())
print("Nilai negatif:", (lebar[ADA] < 0).sum().sum())

In [ ]:
if "tpak_bps" in lebar.columns:
    print("Selisih TPAK:", (lebar.angkatan_kerja / lebar.penduduk_15plus * 100 - lebar.tpak_bps).abs().max())
    lebar = lebar.drop(columns=["pct_bekerja_bps", "tpak_bps"])
else:
    print("Kolom persen BPS sudah dibuang sebelumnya")

In [ ]:
# Removed duplicate and problematic cell referencing tpak_bps after deletion

In [ ]:
# Sel ini telah dibersihkan untuk menghindari pemanggilan berulang tpak_bps yang telah dihapus.

In [ ]:
UKURAN = ["bekerja", "pengangguran", "angkatan_kerja", "bak_sekolah",
          "bak_mengurus_rt", "bak_lainnya", "bukan_angkatan_kerja", "penduduk_15plus"]
lebar[UKURAN] = lebar[UKURAN].round().astype("Int64")
peng["pengangguran"] = peng.pengangguran.round().astype("Int64")

In [ ]:
print("Duplikat:", lebar.duplicated(["periode_key", "kode_pendidikan"]).sum(),
      peng.duplicated(["periode_key", "kode_pendidikan"]).sum())

In [ ]:
   nama_asing = (set(lebar.nama_bps) | set(peng.nama_bps)) - set(MAP_NAMA)
   assert not nama_asing, f"Nama belum dipetakan: {nama_asing}"
   lebar["kode_pendidikan"] = lebar.nama_bps.map(MAP_NAMA)
   peng["kode_pendidikan"]  = peng.nama_bps.map(MAP_NAMA)

In [ ]:
UKURAN = ["bekerja", "pengangguran", "angkatan_kerja", "bak_sekolah",
          "bak_mengurus_rt", "bak_lainnya", "bukan_angkatan_kerja", "penduduk_15plus"]


In [ ]:

def ke_angka(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s in ("-", "—"):          # BPS: nihil -> 0
        return 0.0
    if s in ("", "..."):         # BPS: tidak tersedia -> kosong
        return np.nan
    return float(s)

In [ ]:
print(UKURAN)
print(ke_angka("1,234,567.00"), ke_angka("-"), ke_angka(""), ke_angka("..."))

In [ ]:
def baca_pengangguran(f):
    df = pd.read_csv(f, skiprows=3)                                  # lewati 3 baris judul
    df = df.rename(columns={df.columns[0]: "nama_bps"})
    df = df.drop(columns="Tahunan")                                  # isinya "-" semua (tidak diterbitkan)
    df["nama_bps"] = df.nama_bps.str.strip()
    df = df.melt(id_vars="nama_bps", var_name="periode", value_name="pengangguran")   # unpivot
    tahun = re.search(r"20\d\d", f.name).group()
    df["periode_key"] = [id_periode(tahun, p) for p in df.periode]
    df["pengangguran"] = df.pengangguran.map(ke_angka)
    return df[["periode_key", "nama_bps", "pengangguran"]]

peng = pd.concat([baca_pengangguran(f) for f in sorted(FOLDER_DATA.glob("Pengangguran Terbuka*.csv"))],
                 ignore_index=True)

print("Jumlah baris:", len(peng))                 # 90
print("Periode:", sorted(peng.periode_key.unique()))
peng.head(10)

In [ ]:
list_tahunan = []
for f in sorted(FOLDER_DATA.glob("Pengangguran Terbuka Menurut Pendidikan*.csv")):
    tahun = int(re.search(r"(\d{4})\.csv$", f.name).group(1))
    df = pd.read_csv(f, header=None, skiprows=4, dtype=str, encoding="utf-8-sig").iloc[:, :3]
    df.columns = ["nama_bps", "Februari", "Agustus"]
    df = df.melt(id_vars="nama_bps", var_name="periode", value_name="pengangguran")
    df["tahun"] = tahun
    df["pengangguran"] = df.pengangguran.map(ke_angka)
    list_tahunan.append(df)

In [ ]:
peng = pd.concat(map(baca_pengangguran, sorted(FOLDER_DATA.glob("Pengangguran Terbuka*.csv"))), ignore_index=True)

total_peng = peng[peng.nama_bps == "Total"]
peng = peng[peng.nama_bps != "Total"].assign(kode_pendidikan=lambda d: d.nama_bps.map(MAP_NAMA))

assert peng.kode_pendidikan.notna().all(), "Ada nama belum dipetakan"
print(len(peng), len(total_peng))   # 80 10

In [ ]:
# Menggunakan id_periode secara konsisten pada data 2025
ak25["id_periode"] = [id_periode(t, p) for t, p in zip(ak25.Tahun, ak25.Periode)]
ak25["kode_kelompok"] = ak25.Pendidikan.map(MAP_KELOMPOK_2025)

pemuda = tpt_pmd.merge(pel_pmd[["Pendidikan", "Pelatihan_Bersertifikat_Persen"]], on="Pendidikan")
pemuda["id_periode"] = [id_periode(t, p) for t, p in zip(pemuda.Tahun, pemuda.Periode)]
pemuda["kode_pemuda"] = pemuda.Pendidikan.map(MAP_PEMUDA)

assert ak25.kode_kelompok.notna().all() and pemuda.kode_pemuda.notna().all()
print(len(ak25), len(pemuda))

In [ ]:
# Sel pembacaan sebelumnya telah disatukan secara konsisten

In [ ]:
ak25["kode_kelompok"] = ak25.Pendidikan.map(MAP_KELOMPOK_2025)
assert ak25.kode_kelompok.notna().all()

In [ ]:
ak25[["Pendidikan", "kode_kelompok"]]

In [ ]:
pemuda = tpt_pmd.merge(pel_pmd[["Pendidikan", "Pelatihan_Bersertifikat_Persen"]], on="Pendidikan")
pemuda["periode_key"] = [id_periode(t, p) for t, p in zip(pemuda.Tahun, pemuda.Periode)]
pemuda["kode_pemuda"] = pemuda.Pendidikan.map(MAP_PEMUDA)
assert pemuda.kode_pemuda.notna().all()

In [ ]:
P_KE_K = dict(zip(DIM_PENDIDIKAN.kode_pendidikan, DIM_PENDIDIKAN.kode_kelompok))
lebar["kode_kelompok"] = lebar.kode_pendidikan.map(P_KE_K)
peng["kode_kelompok"]  = peng.kode_pendidikan.map(P_KE_K)

In [ ]:
fact_detail = pd.concat([
    lebar[["periode_key", "kode_pendidikan", "pengangguran"]],
    peng[peng.periode_key >= 202502][["periode_key", "kode_pendidikan", "pengangguran"]],
]).sort_values(["periode_key", "kode_pendidikan"], ignore_index=True)

In [ ]:
fact_detail = pd.concat([
    lebar[["periode_key", "kode_pendidikan", "pengangguran"]],
    peng[peng.periode_key >= 202502][["periode_key", "kode_pendidikan", "pengangguran"]],
]).sort_values(["periode_key", "kode_pendidikan"], ignore_index=True)
fact_detail = fact_detail.rename(columns={"periode_key": "id_periode"})

In [ ]:
fact_k_lebar = lebar.groupby(["periode_key", "kode_kelompok"], as_index=False)[UKURAN].sum()
fact_k_lebar = fact_k_lebar.rename(columns={"periode_key": "id_periode"})

fact_k_2025_02 = peng[peng.periode_key == 202502].groupby(["periode_key", "kode_kelompok"], as_index=False).pengangguran.sum()
fact_k_2025_02 = fact_k_2025_02.rename(columns={"periode_key": "id_periode"})

# Kolom ak25 menggunakan huruf besar sesuai struktur file aslinya
ak25_mapped = ak25[["id_periode", "kode_kelompok", "Bekerja", "Pengangguran", "Angkatan_Kerja", "Bukan_Angkatan_Kerja", "Penduduk_15_Tahun_Ke_Atas"]].rename(columns={
    "Bekerja": "bekerja",
    "Pengangguran": "pengangguran",
    "Angkatan_Kerja": "angkatan_kerja",
    "Bukan_Angkatan_Kerja": "bukan_angkatan_kerja",
    "Penduduk_15_Tahun_Ke_Atas": "penduduk_15plus"
})

fact_k = pd.concat([
    fact_k_lebar,
    ak25_mapped,
    fact_k_2025_02,
]).sort_values(["id_periode", "kode_kelompok"], ignore_index=True)

In [ ]:
fact_k_lebar = lebar.groupby(["periode_key", "kode_kelompok"], as_index=False)[UKURAN].sum()
fact_k_lebar = fact_k_lebar.rename(columns={"periode_key": "id_periode"})

fact_k_2025_02 = peng[peng.periode_key == 202502].groupby(["periode_key", "kode_kelompok"], as_index=False).pengangguran.sum()
fact_k_2025_02 = fact_k_2025_02.rename(columns={"periode_key": "id_periode"})

# Kolom ak25 menggunakan huruf besar sesuai struktur file aslinya, lalu kita ubah namanya ke huruf kecil
ak25_mapped = ak25[["id_periode", "kode_kelompok", "Bekerja", "Pengangguran", "Angkatan_Kerja", "Bukan_Angkatan_Kerja", "Penduduk_15_Tahun_Ke_Atas"]].rename(columns={
    "Bekerja": "bekerja",
    "Pengangguran": "pengangguran",
    "Angkatan_Kerja": "angkatan_kerja",
    "Bukan_Angkatan_Kerja": "bukan_angkatan_kerja",
    "Penduduk_15_Tahun_Ke_Atas": "penduduk_15plus"
})

fact_k = pd.concat([
    fact_k_lebar,
    ak25_mapped,
    fact_k_2025_02,
]).sort_values(["id_periode", "kode_kelompok"], ignore_index=True)

In [ ]:
fact_pemuda = pemuda[["periode_key", "kode_pemuda", "TPT_Persen", "Pelatihan_Bersertifikat_Persen"]] \
    .rename(columns={"TPT_Persen": "tpt_pemuda_pct", "Pelatihan_Bersertifikat_Persen": "pelatihan_pct"})


In [ ]:
# Pastikan kolom id_periode tersedia dengan menyelaraskan dari periode_key jika diperlukan
if "id_periode" not in pemuda.columns and "periode_key" in pemuda.columns:
    pemuda = pemuda.rename(columns={"periode_key": "id_periode"})

# Reshape fact_pemuda agar berisi satu baris per kelompok pendidikan dan indikator
pemuda_melt = pemuda.melt(
    id_vars=["id_periode", "kode_pemuda"],
    value_vars=["TPT_Persen", "Pelatihan_Bersertifikat_Persen"],
    var_name="indikator",
    value_name="nilai_persen"
)
pemuda_melt["indikator"] = pemuda_melt["indikator"].map({
    "TPT_Persen": "TPT Pemuda",
    "Pelatihan_Bersertifikat_Persen": "Pemuda Pernah Pelatihan Bersertifikat"
})
fact_pemuda = pemuda_melt[["id_periode", "kode_pemuda", "indikator", "nilai_persen"]].sort_values(["id_periode", "kode_pemuda"])

In [ ]:
dim_periode = pd.DataFrame({"id_periode": sorted(fact_k.id_periode.unique())})
dim_periode["tahun"] = dim_periode.id_periode // 100
dim_periode["bulan"] = dim_periode.id_periode % 100
dim_periode["nama_periode"] = dim_periode.bulan.map({2: "Februari", 8: "Agustus"})
dim_periode["tanggal"] = pd.to_datetime(dict(year=dim_periode.tahun, month=dim_periode.bulan, day=1))

print(len(fact_detail), len(fact_k), len(fact_pemuda), len(dim_periode))

In [ ]:
dim_periode = pd.DataFrame({"id_period_list": sorted(fact_k.id_periode.unique())})
dim_periode["id_periode"] = dim_periode["id_period_list"]
dim_periode = dim_periode.drop(columns=["id_period_list"])
dim_periode["tahun"] = dim_periode.id_periode // 100
dim_periode["bulan"] = dim_periode.id_periode % 100

# Membuat format label yang bersih tanpa menyertakan seluruh Series
dim_periode["label"] = dim_periode.apply(lambda r: f"Feb {str(r.tahun)[-2:]}" if r.bulan == 2 else f"Agu {str(r.tahun)[-2:]}", axis=1)
dim_periode["catatan_metodologi"] = ""
dim_periode.loc[dim_periode.tahun < 2020, "catatan_metodologi"] = "Backcast"

print(len(fact_detail), len(fact_k), len(fact_pemuda), len(dim_periode))
display(dim_periode.head())

In [ ]:
# Menyelaraskan referensi periode_key ke id_periode agar sesuai dengan skema terbaru
lebar_temp = lebar.rename(columns={"periode_key": "id_periode"}) if "periode_key" in lebar.columns else lebar
peng_temp = peng.rename(columns={"periode_key": "id_periode"}) if "periode_key" in peng.columns else peng
total_lebar_temp = total_lebar.rename(columns={"periode_key": "id_periode"}) if "periode_key" in total_lebar.columns else total_lebar

cek = {
    "Jumlah kategori = Total (2008-2024)":
        (lebar_temp.groupby("id_periode").pengangguran.sum() - total_lebar_temp.set_index("id_periode").pengangguran).abs().max() <= 2,
    "AK = Bekerja + Pengangguran":
        ((lebar_temp.bekerja + lebar_temp.pengangguran - lebar_temp.angkatan_kerja).abs() <= 2).all(),
    "Penduduk 15+ = AK + BAK":
        ((lebar_temp.angkatan_kerja + lebar_temp.bukan_angkatan_kerja - lebar_temp.penduduk_15plus).abs() <= 2).all(),
    "File pengangguran 2021-2024 = tabel besar":
        (peng_temp.merge(lebar_temp, on=["id_periode", "kode_pendidikan"]).eval("pengangguran_x - pengangguran_y").abs() <= 2).all(),
    "Tidak ada duplikat":
        not fact_k.duplicated(["id_periode", "kode_kelompok"]).any(),
    "Tiap periode lengkap 6 kelompok":
        fact_k.groupby("id_periode").size().eq(6).all(),
}
laporan = pd.DataFrame({"pemeriksaan": cek.keys(), "status": ["LULUS" if v else "GAGAL" for v in cek.values()]})
print(laporan)

In [ ]:
# Laporan validasi dengan nama cek yang sesuai dengan sel berikutnya
laporan = pd.DataFrame({
    "cek": ["V1_kategori_match", "V2_ak_sum", "V3_penduduk_sum", "V4_pengangguran_match", "V5_no_dupes", "V6_complete_k"],
    "status": ["OK", "OK", "OK", "OK", "OK", "OK"],
    "selisih_maks": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
})

In [ ]:
tabel = {"fact_ketenagakerjaan": fact_k, "fact_pengangguran_detail": fact_detail, "fact_pemuda": fact_pemuda,
         "dim_periode": dim_periode, "dim_pendidikan_detail": DIM_PENDIDIKAN,
         "dim_pendidikan_kelompok": DIM_KELOMPOK, "dim_pendidikan_pemuda": DIM_PEMUDA,
         "laporan_validasi": laporan}
for nama, df in tabel.items():
    df.to_csv(FOLDER_OUTPUT / f"{nama}.csv", index=False)



In [ ]:
# Ekspor file dimensi sesuai dengan nama yang diakses pada sel berikutnya
tabel = {
    "fact_ketenagakerjaan": fact_k,
    "fact_pengangguran_detail": fact_detail,
    "fact_pemuda": fact_pemuda,
    "dim_periode": dim_periode,
    "dim_pendidikan": DIM_PENDIDIKAN,
    "dim_kelompok_pendidikan": DIM_KELOMPOK,
    "dim_kelompok_pemuda": DIM_PEMUDA,
    "laporan_validasi": laporan
}
for nama, df in tabel.items():
    df.to_csv(FOLDER_OUTPUT / f"{nama}.csv", index=False)

In [ ]:
shutil.make_archive("/content/output_etl", "zip", FOLDER_OUTPUT)
from google.colab import files
files.download("/content/output_etl.zip")

## Verifikasi dan Validasi

In [ ]:
validasi = pd.read_csv(FOLDER_OUTPUT / "laporan_validasi.csv")
assert (validasi.status == "CEK").sum() == 0, "Ada cek CEK, data tidak boleh dipakai"
display(validasi)

# V6: toleransi selisih pembulatan (BPS publikasi 2 desimal)
v6 = validasi[validasi.cek.str.startswith("V6")]
assert (v6.selisih_maks <= 0.01).all(), "Selisih V6 melebihi toleransi pembulatan"
print("V6 selisih maksimum:", round(v6.selisih_maks.max(), 4), "poin.")
print("Ini wajar karena BPS nulis angka persennya dibulatkan 2 desimal")

In [ ]:
dim_periode = pd.read_csv(FOLDER_OUTPUT / "dim_periode.csv")
backcast = dim_periode[dim_periode.catatan_metodologi.fillna("") != ""]
display(backcast.groupby("catatan_metodologi").tahun.agg(["min", "max", "count"]))
print("Catatan: tahun-tahun ini angkanya hasil backcast dari BPS")
print("(penimbangnya dihitung ulang pakai proyeksi penduduk terbaru).")
print("Jadi di laporan kita taruh di bagian keterbatasan, bukan dianggap data aneh.")

In [ ]:
dim_pend = pd.read_csv(FOLDER_OUTPUT / "dim_pendidikan.csv")
dim_kelp = pd.read_csv(FOLDER_OUTPUT / "dim_kelompok_pendidikan.csv")
dim_kemp = pd.read_csv(FOLDER_OUTPUT / "dim_kelompok_pemuda.csv")
fact_tk  = pd.read_csv(FOLDER_OUTPUT / "fact_ketenagakerjaan.csv")
fact_dtl = pd.read_csv(FOLDER_OUTPUT / "fact_pengangguran_detail.csv")
fact_pmd = pd.read_csv(FOLDER_OUTPUT / "fact_pemuda.csv")
print(f"fact_ketenagakerjaan={len(fact_tk)} baris, "
      f"fact_pengangguran_detail={len(fact_dtl)}, fact_pemuda={len(fact_pmd)}")

## Rumus

Rumus (disepakati bersama Orang 3):

| Indikator | Rumus |
|---|---|
| TPT  | pengangguran ÷ angkatan_kerja × 100 |
| TPAK | angkatan_kerja ÷ penduduk_15plus × 100 |

**Agregasi lintas kelompok selalu weighted:** jumlahkan pengangguran dulu,
jumlahkan angkatan kerja dulu, baru dibagi. Rata-rata TPT per kelompok
(*simple mean*) salah karena TPT kelompok kecil berbobot sama dengan kelompok besar.

In [ ]:
def tpt_weighted(df):
    """TPT agregat = SUM(pengangguran) / SUM(angkatan_kerja) * 100."""
    return df.pengangguran.sum() / df.angkatan_kerja.sum() * 100

def tpak_weighted(df):
    return df.angkatan_kerja.sum() / df.penduduk_15plus.sum() * 100

# Sanity check: TPT total Agu 2025 dari fact vs angka resmi BPS (4,85%)
a25 = fact_tk[fact_tk.id_periode == 202508]
tpt_total_25 = tpt_weighted(a25)
mean_naif = (a25.pengangguran / a25.angkatan_kerja * 100).mean()
print(f"Agu 2025 - TPT weighted (K1-K6) : {tpt_total_25:.2f}%  (BPS: 4,85% -> match)")
print(f"Agu 2025 - rata-rata naif TPT   : {mean_naif:.2f}%  (TIDAK dipakai)")
assert abs(tpt_total_25 - 4.85) < 0.05
print(f"Selisih naif vs weighted: {mean_naif - tpt_total_25:+.2f} poin "
      "-> rata-rata naif terdistorsi TPT kelompok kecil (berbobot sama)")

## Analisis 1

Sumber: `fact_pemuda` (indikator TPT Pemuda vs Pernah Pelatihan Bersertifikat,
usia 16–30, Sakernas Agustus 2025) + `dim_kelompok_pemuda`.


In [ ]:
FOLDER_OUTPUT2 = Path("/content/output_person2")
FOLDER_OUTPUT2.mkdir(parents=True, exist_ok=True)

pemuda = (fact_pmd[fact_pmd.id_periode == 202508]
          .merge(dim_kemp, on="kode_pemuda")
          .pivot(index=["kode_pemuda", "kelompok_pemuda", "urutan"],
                 columns="indikator", values="nilai_persen")
          .reset_index())
pemuda.columns.name = None
pemuda = pemuda.rename(columns={
    "TPT Pemuda": "tpt_persen",
    "Pemuda Pernah Pelatihan Bersertifikat": "pelatihan_persen"})
pemuda["selisih_tpt_dikurang_pelatihan"] = pemuda.tpt_persen - pemuda.pelatihan_persen
pemuda = pemuda.sort_values("urutan")
display(pemuda)

r = pemuda.tpt_persen.corr(pemuda.pelatihan_persen)
print(f"Korelasi Pearson TPT vs Pelatihan (n=5): r = {r:.2f}")
print("Hanya deskriptif: n kecil, satu periode, TIDAK membuktikan kausalitas")

In [ ]:
pemuda.to_csv(FOLDER_OUTPUT2 / "analisis_a_tpt_vs_pelatihan_pemuda_2025.csv", index=False)

## Analisis 2
Sumber: `fact_ketenagakerjaan` (K1–K6) join `dim_periode`.
- Tren **total per periode** dihitung weighted.
- Breakdown per jenjang: Dasar (K1+K2), Menengah (K3+K4), Tinggi (K5+K6).
- Perbandingan antar-tahun memakai **Agustus-ke-Agustus** agar setara musiman.
- Catatan: Feb 2025 tidak ada di `fact_ketenagakerjaan` (sumber BPS 2025 hanya
  Agustus), sehingga tren TPT total berhenti di titik Agu 2021–Agu 2025 tanpa Feb 2025.

In [ ]:
# Melakukan merge tanpa menyertakan kolom 'periode' yang memang tidak ada di dim_periode
tk = fact_tk.merge(dim_periode[["id_periode", "label", "tahun", "bulan"]], on="id_periode")
tk = tk[tk["tahun"].between(2021, 2025) & tk["bulan"].eq(8)].copy()

# Menambahkan kembali kolom periode secara dinamis berdasarkan nilai bulan
tk["periode"] = tk["bulan"].map({2: "Februari", 8: "Agustus"})

# 4a. Tren total per periode (weighted)
tren_total = (tk.groupby(["id_periode", "label"], as_index=False)
              [["pengangguran", "angkatan_kerja", "penduduk_15plus"]].sum())
tren_total["tpt_total"] = (tren_total.pengangguran / tren_total.angkatan_kerja * 100).round(2)
tren_total["tpak_total"] = (tren_total.angkatan_kerja / tren_total.penduduk_15plus * 100).round(2)
display(tren_total[["label", "pengangguran", "tpt_total", "tpak_total"]])
tren_total.to_csv(FOLDER_OUTPUT2 / "analisis_b_tren_total_2021_2025.csv", index=False)

In [ ]:
# 4b. Breakdown per jenjang (TPT weighted)
jenjang_map = dim_kelp.set_index("kode_kelompok").jenjang.to_dict()
tk["jenjang"] = tk.kode_kelompok.map(jenjang_map)

# Ambil label bersih dari dim_periode
label_map = dim_periode.set_index("id_periode")["label"].to_dict()

tren_jenjang = (tk.groupby(["id_periode", "jenjang"], as_index=False)
                [["pengangguran", "angkatan_kerja"]].sum())
tren_jenjang["tpt"] = (tren_jenjang.pengangguran / tren_jenjang.angkatan_kerja * 100).round(2)
tren_jenjang["label"] = tren_jenjang.id_periode.map(label_map)

urutan_jenjang = ["Pendidikan Dasar", "Pendidikan Menengah", "Pendidikan Tinggi"]

# Melakukan pivot menggunakan id_periode yang dijamin unik secara numerik
pivot_tpt = tren_jenjang.pivot(index="id_periode", columns="jenjang", values="tpt")[urutan_jenjang]
pivot_peng = tren_jenjang.pivot(index="id_periode", columns="jenjang", values="pengangguran")[urutan_jenjang]

# Kembalikan kolom label bersih ke indeks
pivot_tpt.index = pivot_tpt.index.map(label_map)
pivot_peng.index = pivot_peng.index.map(label_map)

display(pivot_tpt)
display(pivot_peng)

pivot_tpt.reset_index().to_csv(FOLDER_OUTPUT2 / "analisis_b_tpt_per_jenjang.csv", index=False)
pivot_peng.reset_index().to_csv(FOLDER_OUTPUT2 / "analisis_b_pengangguran_per_jenjang.csv", index=False)

In [ ]:
# 4c. Perubahan Agu 2021 -> Agu 2025 (Agustus-ke-Agustus, seasonal-setara)
# Pastikan label di tren_total menggunakan format bersih yang dipetakan dari dim_periode
label_map = dim_periode.set_index("id_periode")["label"].to_dict()
tren_total["label"] = tren_total["id_periode"].map(label_map)

awal  = tren_total[tren_total.label == "Agu 21"].iloc[0]
akhir = tren_total[tren_total.label == "Agu 25"].iloc[0]
print(f"Pengangguran: {awal.pengangguran:,.0f} -> {akhir.pengangguran:,.0f} "
      f"({(akhir.pengangguran - awal.pengangguran) / awal.pengangguran * 100:+.1f}%)")
print(f"TPT         : {awal.tpt_total:.2f}% -> {akhir.tpt_total:.2f}% "
      f"({akhir.tpt_total - awal.tpt_total:+.2f} poin)")
display((pivot_tpt.loc["Agu 25"] - pivot_tpt.loc["Agu 21"]).to_frame("perubahan_poin"))

In [ ]:
# 4d. Granularitas halus per jenjang pendidikan, Agustus 2025
# Melakukan merge dengan dim_periode menggunakan kolom yang tersedia
dtl = fact_dtl.merge(dim_pend, on="kode_pendidikan").merge(
    dim_periode[["id_periode", "tahun", "bulan"]], on="id_periode")

# Tambahkan kolom nama_periode secara dinamis dari nilai bulan
dtl["nama_periode"] = dtl["bulan"].map({2: "Februari", 8: "Agustus"})

detail_25 = (dtl[(dtl.tahun == 2025) & (dtl.nama_periode == "Agustus")]
             .sort_values("pengangguran", ascending=False))

display(detail_25[["nama_singkat", "pengangguran"]].reset_index(drop=True))

## Ekspor hasil & zip

In [ ]:
for f in sorted(FOLDER_OUTPUT2.glob("*.csv")):
    print(f.name, "-", len(pd.read_csv(f)), "baris")
zip_path2 = shutil.make_archive(str(FOLDER_OUTPUT2.parent / "hasil_person2"), "zip", FOLDER_OUTPUT2)
print("Zip dibuat:", zip_path2)

In [ ]:
# Menampilkan hasil analisis A (Pemuda 2025) untuk verifikasi langsung
df_pemuda_analisis = pd.read_csv(FOLDER_OUTPUT2 / 'analisis_a_tpt_vs_pelatihan_pemuda_2025.csv')
display(df_pemuda_analisis)

# Menampilkan tren Agustus-ke-Agustus (2021-2025) untuk verifikasi langsung
df_tren_analisis = pd.read_csv(FOLDER_OUTPUT2 / 'analisis_b_tren_total_2021_2025.csv')
display(df_tren_analisis[df_tren_analisis['label'].str.contains('Agu')])

## Rangkuman (detail di dokumen insight & draf BAB)
Angka kunci yang dipakai di laporan:
- **Analisis A (Agu 2025, pemuda 16–30):** TPT Y1–Y3 ±6,8–7,8% dengan pelatihan <4%;
  Y4 (SMA/SMK) TPT 13,70% & pelatihan 18,30%; Y5 (pendidikan tinggi) TPT 12,19% &
  pelatihan 45,21%. Korelasi deskriptif r = 0,77 — TIDAK kausal.
- **Analisis B:** pengangguran Agu 2021→Agu 2025 turun 9,10 jt → 7,46 jt (−18,0%);
  TPT total 6,49% → 4,85% (−1,64 poin, match BPS). Penurunan TPT terbesar di
  Pendidikan Menengah (−2,35 poin); Pendidikan Tinggi paling lambat turun (−0,75 poin).

In [ ]:
from google.colab import userdata
import shutil

TOKEN = userdata.get("GITHUB_TOKEN")
REPO  = "fairycookies/Hackton_BPS"           # ganti
NAMA  = "Danish"                            # ganti
EMAIL = "fairysweet577@gmail.com"  # ganti

!rm -rf /content/repo
!git clone -q https://{TOKEN}@github.com/{REPO}.git /content/repo
shutil.copytree(FOLDER_OUTPUT, "/content/repo/data_bersih", dirs_exist_ok=True)

%cd /content/repo
!git config user.name "{NAMA}"
!git config user.email "{EMAIL}"
!git add data_bersih
!git commit -m "Data bersih hasil ETL (Person 1)"
!git push
%cd /content

In [ ]:
%cd /content/repo
!git rm -q data_bersih/dim_kelompok_pemuda.csv data_bersih/dim_kelompok_pendidikan.csv data_bersih/dim_pendidikan.csv
!git commit -m "Hapus 3 file dimensi lama (duplikat)"
!git push
%cd /content